# AutoTour — POC reconstruction 3D

Pipeline : **images sélectionnées par `capture-guidee.html`** → **COLMAP** (poses caméra) → **gsplat / nerfstudio (splatfacto)** (Gaussian Splatting) → fichier `.ply` à ouvrir dans `viewer-3d.html`.

Tout ce qui tourne ici est open source et commercialement utilisable :
- **COLMAP** — licence New BSD
- **nerfstudio** (inclut `gsplat`) — licence Apache 2.0

Aucun service tiers (pas de Luma, Polycam, etc.). Ce notebook ne fait qu'exécuter, sur une machine GPU louée temporairement, un pipeline qu'on contrôle et qu'on peut réhéberger n'importe où plus tard.

**Avant de commencer** : dans le menu Colab, `Exécution` → `Modifier le type d'exécution` → sélectionne un accélérateur **GPU**.

## 1. Installation (≈10-15 min, à ne faire qu'une fois par session)

In [ ]:
!nvidia-smi | head -15

# COLMAP (Structure-from-Motion) via apt : suffisant pour ce POC.
# Si c'est trop lent sur vos images, une version compilée avec support CUDA
# ira plus vite — pas nécessaire pour un premier test.
!apt-get -qq update && apt-get -qq install -y colmap

# PyTorch + nerfstudio (qui installe gsplat comme dépendance)
!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/nerfstudio-project/nerfstudio.git

print('Installation terminée.')

## 2. Importer la capture

Choisis le fichier `autotour-capture.zip` exporté depuis `capture-guidee.html` (bouton **« Exporter (test 3D) »**).

In [ ]:
from google.colab import files
import zipfile, os, json, shutil

RAW_DIR = '/content/data/raw'
shutil.rmtree('/content/data', ignore_errors=True)
os.makedirs(RAW_DIR, exist_ok=True)

uploaded = files.upload()  # sélectionner autotour-capture.zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall(RAW_DIR)

with open(os.path.join(RAW_DIR, 'metadata.json')) as f:
    meta = json.load(f)
print(meta['captured_frames'], '/', meta['total_slots'], 'images capturées, exportées le', meta['exported_at'])

img_dir = os.path.join(RAW_DIR, 'images')
n_imgs = len([f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')])
print(n_imgs, 'fichiers image trouvés dans', img_dir)
if n_imgs < 40:
    print('\n\u26a0\ufe0f Peu d\'images pour une reconstruction 3D dense (photogrammétrie/splatting).')
    print('On lance quand même le test : le but est de voir le résultat réel avec cette capture.')

## 3. COLMAP — estimation des poses caméra

`ns-process-data` (nerfstudio) appelle COLMAP en interne : extraction des points d'intérêt, mise en correspondance entre images, puis reconstruction sparse (position + orientation de chaque photo).

In [ ]:
PROCESSED_DIR = '/content/data/processed'
!ns-process-data images --data {img_dir} --output-dir {PROCESSED_DIR} --matching-method sequential

**Si cette étape échoue ou ne trouve pas assez de correspondances** : c'est en général le premier signe que la capture actuelle (36 images, un seul niveau de hauteur) n'a pas assez de recouvrement entre images consécutives, surtout sur une carrosserie peu texturée. C'est une information utile en soi — pas la peine de tout réessayer, notez l'erreur et passez à l'étape 6 (analyse).

## 4. Entraînement du Gaussian Splatting (gsplat, via nerfstudio `splatfacto`)

In [ ]:
!ns-train splatfacto --data {PROCESSED_DIR} --max-num-iterations 10000 \
  --viewer.quit-on-train-completion True --vis viewer

import glob
configs = sorted(glob.glob('/content/outputs/**/splatfacto/**/config.yml', recursive=True))
assert configs, "Aucun config.yml trouvé — l'entraînement n'a probablement pas démarré (voir l'étape 3)."
config_path = configs[-1]
print('Config du modèle entraîné :', config_path)

## 5. Export du modèle en `.ply`

In [ ]:
EXPORT_DIR = '/content/export'
!ns-export gaussian-splat --load-config {config_path} --output-dir {EXPORT_DIR}

ply_files = glob.glob(EXPORT_DIR + '/*.ply')
print('Fichier(s) exporté(s) :', ply_files)
if ply_files:
    files.download(ply_files[0])

## 6. Après le test

Ouvre le `.ply` téléchargé dans `viewer-3d.html` (glisser-déposer le fichier dans la page, ou l'héberger et passer son URL en paramètre `?model=`).

**Nettoyage** : les images brutes de cette session Colab sont supprimées automatiquement quand la session se termine (rien n'est conservé côté serveur au-delà du POC). Pour la suite, seul `.ply` doit être conservé durablement — pas les photos sources.